<a href="https://colab.research.google.com/github/w3aarush/DR_Classification_NIT_MCA_Project/blob/main/EfficientNet_multiclass_SVM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# %pip install tensorflow

In [ ]:
import tensorflow as tf
# from tensorflow.keras.layers.experimental import preprocessing
from tensorflow import keras
from tensorflow.keras import layers,Model
from tensorflow.keras.models import Sequential
from tensorflow.keras.preprocessing.image import ImageDataGenerator, load_img, img_to_array
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.layers import Attention
# from sklearn.svm import SVC
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc, precision_recall_curve, accuracy_score
import matplotlib.pyplot as plt

In [ ]:
# %pip install opencv-python

In [ ]:
# %pip install --extra-index-url=https://pypi.nvidia.com cuml-cu13

In [ ]:
from tensorflow.keras.applications.efficientnet_v2 import EfficientNetV2S, preprocess_input
from google.colab.patches import cv2_imshow
import pandas as pd
import numpy as np
import seaborn as sns
# import imutils
import time
import cv2
from cuml import SVC # for python 3.11
# from sklearn.svm import SVC

In [ ]:
# Install Kaggle API
!pip install -q kaggle

# Upload kaggle.json
from google.colab import files
files.upload()

# Setup Kaggle API
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/kaggle.json
!chmod 600 ~/.kaggle/kaggle.json

# Download DATASET (not competition)
!kaggle datasets download -d mariaherrerot/aptos2019

# Unzip
!unzip -q aptos2019.zip -d aptos2019

# Check files
!ls aptos2019

Saving kaggle.json to kaggle.json
Dataset URL: https://www.kaggle.com/datasets/mariaherrerot/aptos2019
License(s): unknown
100% 8.01G/8.01G [01:08<00:00, 125MB/s]

test.csv  test_images  train_1.csv  train_images  valid.csv  val_images


In [ ]:
base_dir = '/content/aptos2019'
train_dir = '/content/aptos2019/train_images/train_images/'
validation_dir = '/content/aptos2019/val_images/val_images/'
test_dir = '/content/aptos2019/test_images/test_images/'

In [ ]:
import os

In [ ]:
print(os.listdir(train_dir))
print(os.listdir(validation_dir))
print(os.listdir(test_dir))

['2628305cbb29.png', '286e9981dd9b.png', '525d0dd8dc45.png', 'bb733062f494.png', '33105f9b3a04.png', '9287e57326d0.png', '1d46f1326394.png', 'e34fa07bd64d.png', 'a6356a3c5d11.png', '77e15f213b04.png', 'c52bb7343387.png', '655cafb4c932.png', '87d46b1cc4e9.png', 'a44345b27804.png', 'd02b79fc3200.png', 'ab686895533e.png', 'c4e8b1ec8893.png', '9033f1493da1.png', 'd160ebef4117.png', '3d2ecffe0386.png', '42a67337fa8e.png', '7da558d92100.png', 'b90bc89ce8d8.png', '8e4a354e3da2.png', 'cb2f3c5d71a7.png', '48afe8c47454.png', '2bbd1f99ecc3.png', 'b1197f2cc9b3.png', '3f8d5c940ba4.png', 'e1b8acb1cea1.png', '5d4e5fd34d91.png', '37c523296d42.png', '86d58f850a0c.png', 'ba08cee68c71.png', '59fee5bc3479.png', '4e6071b73120.png', '20c883d3bd38.png', '84b4da14bc23.png', 'aad0c0ee9268.png', 'a0adbe677508.png', '7247a2c97f71.png', '64b9206afb3f.png', 'cd45bfa07d41.png', 'a7b7dc8788b9.png', 'd85d052900b4.png', '384631079d1e.png', 'e29e54ff921e.png', '437cbec4a3f8.png', '658ad9f09f5d.png', '494fc9c745a3.png',

In [ ]:
NUM_CLASSES = 5
epochs = 20

In [ ]:
BATCH_SIZE = 32

In [ ]:
img_augmentation = Sequential(
    [
        tf.keras.layers.RandomRotation(factor=(-0.15, 0.15)),
        tf.keras.layers.RandomTranslation(height_factor=0.1, width_factor=0.1),
        tf.keras.layers.RandomFlip(),
        tf.keras.layers.RandomContrast(factor=0.1),
    ],
    name="img_augmentation",
)

In [ ]:
def build_EfficientNetV2S_model_SVM_without_attention(num_classes):
    IMG_SIZE = (224, 224)
    # Load the EfficientNetV2S base model without the top (classification) layer
    base_model = EfficientNetV2S(input_shape=IMG_SIZE + (3,), include_top=False, weights='imagenet')

    # Freeze the base model layers
    base_model.trainable = False

    # Create a new model on top of the base model
    inputs = keras.Input(shape=IMG_SIZE + (3,))
    x = img_augmentation(inputs)
    x = preprocess_input(x) # EfficientNetV2S expects inputs in the [-1, 1] range after preprocessing
    x = base_model(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)
    # outputs = layers.Dense(5, activation='softmax')(x)
    model = keras.Model(inputs, outputs)

    # Compile the model
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

This code defines an image augmentation pipeline using Keras's Sequential model. It applies a series of random transformations to input images:

RandomRotation(factor=(-0.15, 0.15)): Randomly rotates images by an angle within the range of -15% to +15% of 2π radians.
RandomTranslation(height_factor=0.1, width_factor=0.1): Randomly shifts images horizontally and vertically by up to 10% of their width and height, respectively.
RandomFlip(): Randomly flips images horizontally (left-right).
RandomContrast(factor=0.1): Randomly adjusts the contrast of images by a factor within the range of [1 - 0.1, 1 + 0.1] (i.e., [0.9, 1.1]).
This img_augmentation pipeline is typically used during model training to artificially increase the size and diversity of the training dataset, helping to improve the model's generalization capabilities and reduce overfitting.

In [ ]:
def unfreeze(model):
    # unfreeze the top 10 layers while leaving BatchNorm layers frozen for fine-tuning
    for layer in model.layers[-10:]:
        if not isinstance(layer, layers.BatchNormalization):
            layer.trainable = True
    optimizer = tf.keras.optimizers.Adam(learning_rate=1e-4)
    model.compile(optimizer=optimizer, loss='categorical_crossentropy', metrics=['accuracy'])
    '''
    This code defines a function called unfreeze that takes a Keras model as input. Its primary purpose is to prepare a pre-trained model for fine-tuning.

Here's a breakdown:

Unfreezing Top Layers: The for loop iterates through the last 10 layers of the model (model.layers[-10:]). For each of these layers, if it's not a BatchNormalization layer, its trainable attribute is set to True. This means these layers' weights will be updated during subsequent training, allowing for fine-tuning. Batch Normalization layers are often kept frozen during fine-tuning to prevent issues with small batch sizes or drastic changes in statistics.

Optimizer Setup: It initializes an Adam optimizer with a learning rate of 1e-4 (0.0001). Adam is a popular optimization algorithm known for its efficiency.

Model Compilation: Finally, the model is compiled with the specified optimizer, loss='categorical_crossentropy', and metrics=['accuracy']. This prepares the model for training by defining how it will learn (optimizer), what it aims to minimize (loss function, suitable for multi-class classification where labels are one-hot encoded), and what performance indicators to monitor (accuracy).
'''

In [ ]:
def unfreeze_model(model):
    # unfreeze the top 10 layers while leaving BatchNorm layers frozen for fine-tuning
    for layer in model.layers[-10:]:
        if not isinstance(layer, layers.BatchNormalization):
            layer.trainable = True
    optimizer = tf.keras.optimizers.Adam(learning_rate=1e-4)
    model.compile(optimizer=optimizer, loss='categorical_crossentropy', metrics=['accuracy'])

In [ ]:
def test_model(model, test_batches):
    test_labels = test_batches.classes
    print("Test Lables", test_labels)
    print(test_batches.class_indices)

    # predictions = model.predict(test_batches, step=len(test_batches), verbose=0)
    predictions = model.predict(test_batches, verbose=0)


    acc = 0
    for i in range(len(test_labels)):
        actual_class = test_labels[i]
        if predictions[i][actual_class] > 0.5:
            acc += 1
    print('Accuracy:', (acc/len(test_labels))*100, "%")
    print('Classification Report:', classification_report(test_batches.labels, predictions))

In [ ]:
def load_data():
    train = pd.read_csv('/content/aptos2019/train_1.csv', encoding='utf-8')
    test = pd.read_csv('/content/aptos2019/test.csv', encoding='utf-8')
    valid = pd.read_csv('/content/aptos2019/valid.csv')

    train_dir = '/content/aptos2019/train_images/train_images/'
    test_dir = '/content/aptos2019/test_images/test_images/'
    valid_dir = '/content/aptos2019/val_images/val_images/'

    # construct file paths directly within function:
    train['image_path'] = train_dir + train['id_code'] + '.png'
    test['image_path'] = test_dir + test['id_code'] + '.png'
    valid['image_path'] = valid_dir + valid['id_code'] + '.png'

    train['train_images'] = train['id_code'] + '.png'
    test['test_images'] = test['id_code'] + '.png'
    valid['valid_images'] = valid['id_code'] + '.png'

    train['diagnosis'] = train['diagnosis'].astype(str)
    # train['target'] = [1 if x >= 1 else 0 for x in train['diagnosis']]
    # train['target'] = train['target'].astype(str)
    test['diagnosis'] = test['diagnosis'].astype(str)
    # test['target'] = [1 if x >= 1 else 0 for x in test['diagnosis']]
    # test['target'] = test['target'].astype(str)
    valid['diagnosis'] = valid['diagnosis'].astype(str)
    # valid['target'] = [1 if x >= 1 else 0 for x in valid['diagnosis']]
    # valid['target'] = valid['target'].astype(str)

    return train, test, valid

In [ ]:
def preprocess_image(image_path):
    img = load_img(image_path, target_size=(224, 224))
    img_array = img_to_array(img)
    img_array = np.expand_dims(img_array, axis = 0)
    return preprocess_input(img_array)

In [ ]:
if __name__ == "__main__":
    train_df, test_df, valid_df = load_data()
    model = build_EfficientNetV2S_model_SVM_without_attention(NUM_CLASSES)
    unfreeze_model(model)

    train_batches = ImageDataGenerator(preprocessing_function=tf.keras.applications.efficientnet_v2.preprocess_input).flow_from_dataframe(
        dataframe=train_df, x_col='image_path', y_col='diagnosis', target_size=(224,224), batch_size=BATCH_SIZE)
    valid_batches = ImageDataGenerator(preprocessing_function=tf.keras.applications.efficientnet_v2.preprocess_input).flow_from_dataframe(
        dataframe=valid_df, x_col='image_path', y_col='diagnosis', target_size=(224,224), batch_size=BATCH_SIZE)
    test_batches = ImageDataGenerator(preprocessing_function=tf.keras.applications.efficientnet_v2.preprocess_input).flow_from_dataframe(
        dataframe=test_df, x_col='image_path', y_col='diagnosis', target_size=(224,224), batch_size=BATCH_SIZE, shuffle=False)

    early_stopping = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)
    reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=3, min_lr=1e-4)
    history = model.fit(train_batches, epochs=epochs, validation_data=valid_batches, verbose=1,callbacks=[early_stopping,reduce_lr])

82420632/82420632 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step
Found 2930 validated image filenames belonging to 5 classes.
Found 366 validated image filenames belonging to 5 classes.
Found 366 validated image filenames belonging to 5 classes.
Epoch 1/20
92/92 ━━━━━━━━━━━━━━━━━━━━ 492s 4s/step - accuracy: 0.6983 - loss: 0.8479 - val_accuracy: 0.7404 - val_loss: 0.7234 - learning_rate: 1.0000e-04
Epoch 2/20
92/92 ━━━━━━━━━━━━━━━━━━━━ 378s 4s/step - accuracy: 0.7986 - loss: 0.5554 - val_accuracy: 0.7596 - val_loss: 0.6371 - learning_rate: 1.0000e-04
Epoch 3/20
92/92 ━━━━━━━━━━━━━━━━━━━━ 377s 4s/step - accuracy: 0.8212 - loss: 0.4887 - val_accuracy: 0.7787 - val_loss: 0.5602 - learning_rate: 1.0000e-04
Epoch 4/20
92/92 ━━━━━━━━━━━━━━━━━━━━ 376s 4s/step - accuracy: 0.8396 - loss: 0.4338 - val_accuracy: 0.7978 - val_loss: 0.5500 - learning_rate: 1.0000e-04
Epoch 5/20
92/92 ━━━━━━━━━━━━━━━━━━━━ 375s 4s/step - accuracy: 0.8539 - loss: 0.3867 - val_accuracy: 0.7951 - val_loss: 0.5406 - learning_rate: 1.00

First, we need to extract all images and their corresponding labels from the `ImageDataGenerator` objects (`train_batches` and `test_batches`). The `ImageDataGenerator` provides data in batches, so we'll iterate through them and collect all samples.

In [ ]:
import numpy as np

def get_all_data_and_labels(generator):
    all_images = []
    all_labels = []
    for _ in range(len(generator)):
        images, labels = next(generator)
        all_images.append(images)
        all_labels.append(labels)
    return np.vstack(all_images), np.vstack(all_labels)

X_train_data, y_train_onehot = get_all_data_and_labels(train_batches)
X_test_data, y_test_onehot = get_all_data_and_labels(test_batches)

print(f"Shape of X_train_data: {X_train_data.shape}")
print(f"Shape of y_train_onehot: {y_train_onehot.shape}")
print(f"Shape of X_test_data: {X_test_data.shape}")
print(f"Shape of y_test_onehot: {y_test_onehot.shape}")

Now, we will create a feature extraction model from the trained EfficientNetV2S model. The features for the SVM should be the output of the `GlobalAveragePooling2D` layer, which comes before the final classification `Dense` layer. We will also prepare the labels for the SVM by converting the one-hot encoded labels to integer labels.

# load model

In [ ]:
# model.save('my_trained_model.keras')

loaded_model = tf.keras.models.load_model('/content/drive/MyDrive/ML Projects/trained_models/effnet_svm_multiclass_softmax_model_01.keras')

print("Model saved and loaded successfully!")
loaded_model.summary()

Model saved and loaded successfully!


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ img_augmentation (Sequential)   │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ efficientnetv2-s (Functional)   │ (None, 7, 7, 1280)     │    20,331,360 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 5)              │         6,405 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 60,705,553 (231.57 MB)

 Trainable params: 20,183,893 (77.00 MB)

 Non-trainable params: 153,872 (601.06 KB)

 Optimizer params: 40,367,788 (153.99 MB)

In [ ]:
model = loaded_model

# DataFrame

In [ ]:
train_df, test_df, valid_df = load_data()

train_batches = ImageDataGenerator(preprocessing_function=tf.keras.applications.efficientnet_v2.preprocess_input).flow_from_dataframe(dataframe=train_df, x_col='image_path', y_col='diagnosis', target_size=(224,224), batch_size=BATCH_SIZE)

valid_batches = ImageDataGenerator(preprocessing_function=tf.keras.applications.efficientnet_v2.preprocess_input).flow_from_dataframe(dataframe=valid_df, x_col='image_path', y_col='diagnosis', target_size=(224,224), batch_size=BATCH_SIZE)

test_batches = ImageDataGenerator(preprocessing_function=tf.keras.applications.efficientnet_v2.preprocess_input).flow_from_dataframe(dataframe=test_df, x_col='image_path', y_col='diagnosis', target_size=(224,224), batch_size=BATCH_SIZE, shuffle=False)

Found 2930 validated image filenames belonging to 5 classes.
Found 366 validated image filenames belonging to 5 classes.
Found 366 validated image filenames belonging to 5 classes.


# SVM

In [ ]:
X_train, y_train = [], []
for _ in range(len(train_batches)):
    images, labels = next(train_batches)
    X_train.append(images)
    y_train.append(labels)

In [ ]:
# Convert lists to arrays
X_train = np.vstack(X_train)
y_train = np.vstack(y_train)

In [ ]:
X_train.shape

(2930, 224, 224, 3)

In [ ]:
# Reshape X_train to be 2D (necessary for SVM)
X_train1 = X_train.reshape(X_train.shape[0], -1)

In [ ]:
X_train1.shape

(2930, 150528)

In [ ]:
y_train.shape

(2930, 5)

In [ ]:
# Collect data from the generator
X_test, y_test = [], []
for _ in range(len(test_batches)):
    images, labels = next(test_batches)
    X_test.append(images)
    y_test.append(labels)

In [ ]:
# Convert lists to arrays
X_test = np.vstack(X_test)
y_test = np.vstack(y_test)

In [ ]:
X_test.shape

(366, 224, 224, 3)

In [ ]:
# Reshape X_train to be 2D (necessary for SVM)
X_test1 = X_test.reshape(X_test.shape[0], -1)

In [ ]:
X_test1.shape

(366, 150528)

In [ ]:
y_test

array([[0., 0., 0., 0., 1.],
       [1., 0., 0., 0., 0.],
       [1., 0., 0., 0., 0.],
       ...,
       [0., 0., 0., 0., 1.],
       [1., 0., 0., 0., 0.],
       [1., 0., 0., 0., 0.]], dtype=float32)

In [ ]:
# feature_extractor_model = tf.keras.Model(inputs=model.input, outputs=model.get_layer('global_average_pooling2d').output)

# features_array_train = feature_extractor_model.predict(X_train_data)
# features_array_test = feature_extractor_model.predict(X_test_data)

# Convert one-hot encoded labels to integer labels for SVC
y_train_labels = np.argmax(y_train_onehot, axis=1)
y_test_labels = np.argmax(y_test_onehot, axis=1)

# print(f"Shape of extracted training features: {features_array_train.shape}")
# print(f"Shape of extracted test features: {features_array_test.shape}")
print(f"Shape of training labels for SVM: {y_train_labels.shape}")
print(f"Shape of test labels for SVM: {y_test_labels.shape}")

In [ ]:
from tensorflow.python.keras.models import Model
model1 = tf.keras.Model(inputs = model.input, outputs=model.layers[-2].output)

features_array_train1 = model1.predict(X_train)
features_array_train1_reshape = features_array_train1.reshape(features_array_train1.shape[0], -1)

92/92 ━━━━━━━━━━━━━━━━━━━━ 24s 207ms/step


In [ ]:
features_array_test1 = model1.predict(X_test)
features_array_test1_reshape = features_array_test1.reshape(features_array_test1.shape[0], -1)
X_test1 = X_test.reshape(X_test.shape[0], -1)

12/12 ━━━━━━━━━━━━━━━━━━━━ 2s 184ms/step


In [ ]:
svm_clf = SVC(verbose=True, kernel='rbf')
svm_clf.fit(features_array_train1_reshape, y_train.argmax(axis=1))

[2026-04-28 19:18:53.275] [CUML] [debug] Creating working set with 1024 elements
[2026-04-28 19:18:53.555] [CUML] [debug] SMO solver finished after 6 outer iterations, total inner 327 iterations, and diff 0.000991
[2026-04-28 19:18:53.904] [CUML] [debug] Creating working set with 1024 elements
[2026-04-28 19:18:53.944] [CUML] [debug] SMO solver finished after 6 outer iterations, total inner 278 iterations, and diff 0.000995
[2026-04-28 19:18:53.979] [CUML] [debug] Creating working set with 1024 elements
[2026-04-28 19:18:53.996] [CUML] [debug] SMO solver finished after 5 outer iterations, total inner 167 iterations, and diff 0.000991
[2026-04-28 19:18:54.051] [CUML] [debug] Creating working set with 1024 elements
[2026-04-28 19:18:54.127] [CUML] [debug] SMO solver finished after 6 outer iterations, total inner 201 iterations, and diff 0.000997
[2026-04-28 19:18:54.165] [CUML] [debug] Creating working set with 1024 elements
[2026-04-28 19:18:54.167] [CUML] [debug] Could not fill working

SVC()

In [ ]:
pred_1 = svm_clf.predict(features_array_test1_reshape)

In [ ]:
print(accuracy_score(y_test.argmax(axis=1), pred_1))

0.8278688524590164


Now that the SVM is trained, we can evaluate its performance on the test set.

In [ ]:
print(classification_report(test_batches.labels, pred_1))

              precision    recall  f1-score   support

           0       0.56      0.56      0.56       199
           1       0.04      0.03      0.04        30
           2       0.28      0.36      0.31        87
           3       0.07      0.06      0.06        17
           4       0.06      0.03      0.04        33

    accuracy                           0.40       366
   macro avg       0.20      0.21      0.20       366
weighted avg       0.38      0.40      0.39       366



In [ ]:
model.predict(test_batches)

12/12 ━━━━━━━━━━━━━━━━━━━━ 50s 4s/step


array([[9.97072339e-01, 1.09286210e-03, 1.49521336e-04, 9.32359486e-04,
        7.52885535e-04],
       [2.03824881e-03, 3.06009740e-01, 6.77081287e-01, 9.17512923e-03,
        5.69563918e-03],
       [5.10108133e-04, 5.08429203e-03, 1.90372784e-02, 4.45897430e-01,
        5.29470861e-01],
       ...,
       [1.03347775e-04, 5.65072894e-03, 1.92753047e-01, 5.10276437e-01,
        2.91216403e-01],
       [9.96513903e-01, 1.36042410e-03, 6.26412802e-05, 1.39774196e-03,
        6.65377942e-04],
       [1.44490239e-03, 8.65870342e-02, 7.68133521e-01, 9.29157585e-02,
        5.09188324e-02]], dtype=float32)

In [ ]:
test_model(model, test_batches)

Test Lables [0, 2, 4, 0, 0, 0, 3, 2, 1, 1, 0, 0, 2, 0, 2, 0, 0, 0, 0, 4, 0, 0, 2, 2, 2, 0, 0, 0, 0, 0, 2, 0, 2, 2, 2, 0, 4, 2, 0, 3, 0, 0, 2, 2, 0, 4, 2, 2, 2, 0, 4, 0, 2, 2, 2, 0, 3, 0, 0, 0, 2, 1, 0, 3, 2, 0, 0, 1, 0, 4, 0, 0, 2, 0, 1, 2, 1, 2, 0, 2, 2, 0, 0, 0, 0, 0, 2, 4, 0, 0, 1, 4, 2, 0, 2, 0, 0, 2, 4, 2, 0, 1, 0, 4, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 4, 2, 0, 0, 4, 0, 0, 3, 0, 2, 0, 0, 0, 2, 2, 4, 2, 0, 0, 0, 2, 2, 0, 1, 0, 1, 0, 1, 0, 0, 4, 0, 0, 0, 0, 2, 0, 1, 0, 0, 0, 2, 0, 4, 0, 0, 4, 0, 0, 0, 4, 0, 0, 0, 2, 3, 0, 3, 0, 0, 0, 2, 1, 0, 4, 2, 0, 2, 0, 4, 2, 0, 2, 0, 0, 0, 0, 4, 2, 0, 2, 0, 0, 0, 0, 2, 0, 2, 0, 2, 0, 2, 0, 0, 0, 2, 2, 1, 1, 0, 0, 0, 0, 2, 0, 0, 0, 0, 0, 4, 1, 1, 0, 0, 3, 0, 0, 2, 0, 4, 0, 3, 0, 4, 2, 2, 0, 0, 3, 0, 0, 0, 2, 4, 0, 4, 3, 1, 0, 0, 0, 0, 4, 1, 0, 1, 0, 0, 0, 2, 1, 2, 0, 2, 4, 0, 0, 0, 4, 0, 0, 2, 3, 0, 3, 0, 2, 0, 0, 2, 2, 0, 0, 0, 2, 2, 0, 2, 0, 4, 0, 2, 0, 0, 0, 2, 0, 3, 4, 1, 0, 0, 0, 2, 0, 0, 0, 2, 2, 2, 2, 0, 0, 0, 0, 1, 0, 2, 3, 0, 4, 0, 0, 2, 0, 

ValueError: Classification metrics can't handle a mix of multiclass and continuous-multioutput targets

In [ ]:
model.evaluate(test_batches)

12/12 ━━━━━━━━━━━━━━━━━━━━ 58s 3s/step - accuracy: 0.7978 - loss: 0.5546


[0.5545821189880371, 0.7978141903877258]

In [ ]:
model.evaluate(test_batches)

12/12 ━━━━━━━━━━━━━━━━━━━━ 53s 4s/step - accuracy: 0.7978 - loss: 0.5546


[0.5545821189880371, 0.7978141903877258]

In [ ]:
import pickle
with open('svm_model.pkl', 'wb') as file:
    pickle.dump(svm_clf, file)